# 05. Point-in-Time Feature Engineering

## Objective

This notebook creates the first modeling dataset. One row represents one customer who is active at one monthly `ReferenceDate`. Every feature uses only information strictly before that date. The target is `WillChurnNext30Days`: whether the customer reaches 60 consecutive days without a valid purchase during the following 30 days.

## Work plan

1. Load and validate the temporal inputs from notebook 3.
2. Keep the active customer-reference population used by the churn model.
3. Construct cancellation-aware RFM and purchase-rhythm features.
4. Inspect each feature's relationship with the target.
5. Export the validated first feature table.

Purchase rhythm, cancellation behavior, basket profile, and any transformation will be added later, one family at a time, only after its value is evaluated.

## Temporal contract

For a snapshot at time `R`, a purchase can contribute to a feature only when `PurchaseDate < R`. A fully reversed purchase contributes only while its cancellation is still unknown, meaning `FullCancellationDate` is missing or later than `R`. `ChurnEventDate`, `LabelEndDate`, `OutcomeEndDate`, and the target are audit or outcome columns, never predictive features. Learned preprocessing must later be fitted separately inside each temporal training fold.

## 1. Load validated modeling inputs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

snapshots = pd.read_parquet(
    "../data/interim/labeled_customer_snapshots.parquet"
)
purchase_events = pd.read_parquet(
    "../data/interim/purchase_invoice_events.parquet"
)
cancellation_events = pd.read_parquet(
    "../data/interim/cancellation_invoice_events.parquet"
)
customer_transactions = pd.read_parquet(
    "../data/interim/customer_transactions.parquet"
)
fold_date_columns = ["TrainStart", "ValidationStart", "ValidationEnd"]
model_selection_folds = pd.read_csv(
    "../data/interim/model_selection_folds.csv",
    parse_dates=fold_date_columns,
)

print("Snapshots:", snapshots.shape)
print("Purchase events:", purchase_events.shape)
print("Cancellation events:", cancellation_events.shape)
print("Transaction lines:", customer_transactions.shape)
print("Model-selection folds:", model_selection_folds.shape)

### 1.1 Input checks

A zero in every row means that notebook 5 received structurally valid inputs.

In [ ]:
input_checks = pd.Series({
    "Duplicate customer-reference rows": snapshots.duplicated(
        ["Customer ID", "ReferenceDate"]
    ).sum(),
    "Duplicate purchase invoices": purchase_events.duplicated(
        ["Customer ID", "Invoice"]
    ).sum(),
    "Duplicate cancellation invoices": cancellation_events.duplicated(
        ["Customer ID", "Invoice"]
    ).sum(),
    "Missing reference dates": snapshots["ReferenceDate"].isna().sum(),
    "Missing purchase dates": purchase_events["PurchaseDate"].isna().sum(),
    "Invalid fold separation": (~model_selection_folds["NoLabelOverlap"]).sum(),
}, name="Errors").to_frame()
input_checks

## 2. Modeling population

The churn model acts before the churn event, so only customers active at the reference date are model observations. Customers already churned remain in the snapshot export for population analysis but are not included here. Their target is intentionally missing.

In [ ]:
model_snapshots = snapshots.loc[
    ~snapshots["IsChurnedAtReference"]
].copy()
model_snapshots["WillChurnNext30Days"] = (
    model_snapshots["WillChurnNext30Days"].astype(bool)
)

print("Modeling rows:", len(model_snapshots))
print("Customers:", model_snapshots["Customer ID"].nunique())
print("Reference dates:", model_snapshots["ReferenceDate"].nunique())
development_target_rate = model_snapshots.loc[
    model_snapshots["FinalSplit"].eq("development"),
    "WillChurnNext30Days",
].mean() * 100
print("Development target rate:", round(development_target_rate, 2), "%")

## 3. Valid historical purchases

Snapshots are joined to their customer's invoices, then filtered to purchases available at the scoring time. This is an intermediate long table: the same invoice can appear for several later reference dates because it belongs to the history of each of those snapshots.

In [ ]:
snapshot_purchase_history = (
    model_snapshots[["Customer ID", "ReferenceDate"]]
    .merge(purchase_events, on="Customer ID")
)
snapshot_purchase_history = snapshot_purchase_history.loc[
    snapshot_purchase_history["PurchaseDate"].lt(
        snapshot_purchase_history["ReferenceDate"]
    )
    & (
        snapshot_purchase_history["FullCancellationDate"].isna()
        | snapshot_purchase_history["FullCancellationDate"].gt(
            snapshot_purchase_history["ReferenceDate"]
        )
    )
]

print("Historical snapshot-invoice rows:", len(snapshot_purchase_history))
snapshot_purchase_history.head()

In [ ]:
history_checks = pd.Series({
    "Purchases on or after reference date": (
        snapshot_purchase_history["PurchaseDate"]
        >= snapshot_purchase_history["ReferenceDate"]
    ).sum(),
    "Full cancellations known at reference date": (
        snapshot_purchase_history["FullCancellationDate"].notna()
        & (
            snapshot_purchase_history["FullCancellationDate"]
            <= snapshot_purchase_history["ReferenceDate"]
        )
    ).sum(),
}, name="Errors").to_frame()
history_checks

## 4. Point-in-time RFM features

RFM means Recency, Frequency, and Monetary value. `RecencyDays` already comes from the snapshot construction. `PurchaseFrequency` counts distinct valid historical invoices. `MonetaryValue` sums their invoice values. `AverageInvoiceValue` separates total customer value from typical basket value.

In [ ]:
rfm_features = (
    snapshot_purchase_history
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(
        PurchaseFrequency=("Invoice", "nunique"),
        MonetaryValue=("InvoiceValue", "sum"),
    )
)
rfm_features["AverageInvoiceValue"] = (
    rfm_features["MonetaryValue"] / rfm_features["PurchaseFrequency"]
)
snapshot_features = model_snapshots.merge(
    rfm_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)

snapshot_features[[
    "Customer ID", "ReferenceDate", "RecencyDays",
    "PurchaseFrequency", "MonetaryValue", "AverageInvoiceValue"
]].head()

In [ ]:
rfm_checks = pd.Series({
    "Rows added or removed": abs(len(snapshot_features) - len(model_snapshots)),
    "Duplicate customer-reference rows": snapshot_features.duplicated(
        ["Customer ID", "ReferenceDate"]
    ).sum(),
    "Missing RFM values": snapshot_features[[
        "PurchaseFrequency", "MonetaryValue", "AverageInvoiceValue"
    ]].isna().any(axis=1).sum(),
    "Non-positive frequencies": snapshot_features["PurchaseFrequency"].le(0).sum(),
    "Inconsistent average values": (
        snapshot_features["AverageInvoiceValue"]
        * snapshot_features["PurchaseFrequency"]
        - snapshot_features["MonetaryValue"]
    ).abs().gt(0.01).sum(),
}, name="Errors").to_frame()
rfm_checks

### 4.1 Manual example

Customer `12347` illustrates why an invoice can appear in several intermediate rows. Frequency and monetary value accumulate only after an invoice becomes historical relative to each reference date.

In [ ]:
example_customer = "12347"
display(
    purchase_events.loc[
        purchase_events["Customer ID"].eq(example_customer),
        ["Invoice", "PurchaseDate", "InvoiceValue", "FullCancellationDate"],
    ].sort_values("PurchaseDate")
)
snapshot_features.loc[
    snapshot_features["Customer ID"].eq(example_customer),
    [
        "ReferenceDate", "LastPurchaseDate", "RecencyDays",
        "PurchaseFrequency", "MonetaryValue", "AverageInvoiceValue",
        "WillChurnNext30Days",
    ],
].sort_values("ReferenceDate")

### 4.2 Purchase rhythm

`MedianPurchaseGapDays` is the customer's median number of days between distinct historical purchase days available at each reference date. Purchases made on the same calendar day are treated as one purchase occasion so that several invoices from the same session do not create artificial zero-day gaps.

`RecencyToMedianGapRatio` compares current inactivity with this personal rhythm. A ratio above 1 means that the customer has already been inactive longer than their typical historical interval. The ratio is unavailable until at least two distinct purchase days have been observed.

In [ ]:
purchase_day_history = snapshot_purchase_history[[
    "Customer ID", "ReferenceDate", "PurchaseDate"
]].copy()
purchase_day_history["PurchaseDay"] = (
    purchase_day_history["PurchaseDate"].dt.normalize()
)
purchase_day_history = (
    purchase_day_history
    .drop_duplicates(["Customer ID", "ReferenceDate", "PurchaseDay"])
    .sort_values(["Customer ID", "ReferenceDate", "PurchaseDay"])
)
purchase_day_history["PurchaseGapDays"] = (
    purchase_day_history
    .groupby(["Customer ID", "ReferenceDate"])["PurchaseDay"]
    .diff()
    .dt.days
)
rhythm_features = (
    purchase_day_history
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(
        ObservedPurchaseDays=("PurchaseDay", "nunique"),
        MedianPurchaseGapDays=("PurchaseGapDays", "median"),
    )
)
snapshot_features = snapshot_features.merge(
    rhythm_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)
snapshot_features["RecencyToMedianGapRatio"] = (
    snapshot_features["RecencyDays"]
    / snapshot_features["MedianPurchaseGapDays"]
)

snapshot_features[[
    "Customer ID", "ReferenceDate", "RecencyDays",
    "ObservedPurchaseDays", "MedianPurchaseGapDays",
    "RecencyToMedianGapRatio",
]].head(10)

In [ ]:
rhythm_checks = pd.Series({
    "Rows added or removed": abs(len(snapshot_features) - len(model_snapshots)),
    "Missing ratio with at least two purchase days": snapshot_features.loc[
        snapshot_features["ObservedPurchaseDays"].ge(2),
        "RecencyToMedianGapRatio",
    ].isna().sum(),
    "Ratio present with fewer than two purchase days": snapshot_features.loc[
        snapshot_features["ObservedPurchaseDays"].lt(2),
        "RecencyToMedianGapRatio",
    ].notna().sum(),
    "Non-positive median gaps": snapshot_features[
        "MedianPurchaseGapDays"
    ].le(0).sum(),
}, name="Errors").to_frame()
rhythm_checks

### 4.3 Product-description inventory before categorization

This inspection asks whether valid purchase descriptions can support understandable product categories. It includes positive-quantity lines from non-cancellation invoices and excludes manual accounting lines. Zero-price gifts remain included because they are valid customer-product interactions. No category or predictive feature is created yet.

Descriptions are converted to uppercase and repeated spaces are removed only to consolidate superficial text variants. Service-like descriptions such as postage are intentionally left visible so they can be identified before deciding what constitutes a product category.

In [ ]:
product_lines = customer_transactions.loc[
    ~customer_transactions["Invoice"].str.startswith("C")
    & customer_transactions["Quantity"].gt(0)
    & customer_transactions["StockCode"].ne("M"),
    ["Description", "StockCode", "Invoice", "Customer ID", "Quantity"],
].copy()
product_lines["Description"] = (
    product_lines["Description"]
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

product_description_inventory = (
    product_lines.groupby("Description", as_index=False)
    .agg(
        StockCodes=("StockCode", "nunique"),
        PurchaseLines=("Invoice", "size"),
        Invoices=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique"),
        Units=("Quantity", "sum"),
    )
    .sort_values(["Invoices", "Customers"], ascending=False)
)

print("Purchase lines inspected:", len(product_lines))
print("Distinct descriptions:", len(product_description_inventory))
display(product_description_inventory.head(50))

The notebook displays the 50 descriptions appearing in the most invoices. The complete alphabetical inventory is exported to CSV because thousands of rows cannot be inspected reliably in a truncated notebook display.

In [ ]:
description_inventory_path = (
    "../data/interim/product_description_inventory.csv"
)
product_description_inventory.sort_values("Description").to_csv(
    description_inventory_path, index=False
)
print("Saved:", description_inventory_path)

### 4.4 Conservative Christmas category

The first product category uses explicit seasonal terms: `CHRISTMAS`, `XMAS`, `SANTA`, `NOEL`, `ADVENT`, `REINDEER`, `MISTLETOE`, and `NATIVITY`. More ambiguous terms such as `HOLLY`, `SNOWFLAKE`, and `BAUBLE` are audited separately but excluded from the feature for now. This prioritizes category precision over maximum coverage.

In [ ]:
christmas_pattern = (
    r"CHRISTMAS|XMAS|\bSANTAS?\b|\bNOEL\b|\bADVENT\b|"
    r"\bREINDEER\b|\bMISTLETOE\b|\bNATIVITY\b"
)
ambiguous_christmas_pattern = (
    r"\bSNOWMAN\b|\bSNOWFLAKES?\b|\bHOLLY\b|\bBAUBLES?\b"
)
product_lines["IsChristmasProduct"] = (
    product_lines["Description"].str.contains(
        christmas_pattern, case=False, na=False
    )
)
strict_christmas_descriptions = product_description_inventory.loc[
    product_description_inventory["Description"].str.contains(
        christmas_pattern, case=False, na=False
    )
].sort_values("Invoices", ascending=False)
ambiguous_christmas_descriptions = product_description_inventory.loc[
    product_description_inventory["Description"].str.contains(
        ambiguous_christmas_pattern, case=False, na=False
    )
    & ~product_description_inventory["Description"].str.contains(
        christmas_pattern, case=False, na=False
    )
].sort_values("Invoices", ascending=False)

print("Strict Christmas descriptions:", len(strict_christmas_descriptions))
print("Ambiguous descriptions excluded:", len(ambiguous_christmas_descriptions))
display(strict_christmas_descriptions.head(50))
display(ambiguous_christmas_descriptions)

### 4.5 Historical Christmas invoice share

`ChristmasInvoiceShare` is the share of valid historical invoices containing at least one strictly classified Christmas product. Counting invoices rather than lines or units prevents large baskets from dominating the feature. The existing cancellation-aware snapshot history guarantees that only invoices valid and known before each reference date contribute.

In [ ]:
invoice_product_profile = (
    product_lines.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(HasChristmasProduct=("IsChristmasProduct", "max"))
)
snapshot_product_history = (
    snapshot_purchase_history[["Customer ID", "ReferenceDate", "Invoice"]]
    .merge(
        invoice_product_profile,
        on=["Customer ID", "Invoice"],
        how="left",
        validate="many_to_one",
    )
)
christmas_features = (
    snapshot_product_history
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(
        HistoricalInvoices=("Invoice", "nunique"),
        ChristmasInvoices=("HasChristmasProduct", "sum"),
    )
)
christmas_features["ChristmasInvoiceShare"] = (
    christmas_features["ChristmasInvoices"]
    / christmas_features["HistoricalInvoices"]
)
snapshot_features = snapshot_features.merge(
    christmas_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)

snapshot_features[[
    "Customer ID", "ReferenceDate", "PurchaseFrequency",
    "ChristmasInvoices", "ChristmasInvoiceShare",
]].head(10)

In [ ]:
christmas_checks = pd.Series({
    "Rows added or removed": abs(len(snapshot_features) - len(model_snapshots)),
    "Missing Christmas shares": snapshot_features[
        "ChristmasInvoiceShare"
    ].isna().sum(),
    "Historical invoice count mismatches": (
        snapshot_features["HistoricalInvoices"]
        != snapshot_features["PurchaseFrequency"]
    ).sum(),
    "Shares outside 0 to 1": (
        ~snapshot_features["ChristmasInvoiceShare"].between(0, 1)
    ).sum(),
}, name="Errors").to_frame()
christmas_checks

### 4.6 Historical product breadth

`UniqueProductsPurchased` counts distinct physical `StockCode` values in the valid purchase history available at each reference date. Manual lines are excluded because they are accounting entries rather than identifiable products. The model will test whether product breadth adds information after purchase frequency is already controlled.

In [ ]:
snapshot_product_lines = (
    snapshot_purchase_history[["Customer ID", "ReferenceDate", "Invoice"]]
    .merge(
        product_lines[["Customer ID", "Invoice", "StockCode"]]
        .drop_duplicates(),
        on=["Customer ID", "Invoice"],
        how="left",
        validate="many_to_many",
    )
)
product_breadth_features = (
    snapshot_product_lines
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(UniqueProductsPurchased=("StockCode", "nunique"))
)
snapshot_features = snapshot_features.merge(
    product_breadth_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)

snapshot_features[[
    "Customer ID", "ReferenceDate", "PurchaseFrequency",
    "UniqueProductsPurchased",
]].head(10)

In [ ]:
product_breadth_checks = pd.Series({
    "Rows added or removed": abs(len(snapshot_features) - len(model_snapshots)),
    "Missing product counts": snapshot_features[
        "UniqueProductsPurchased"
    ].isna().sum(),
    "Negative product counts": snapshot_features[
        "UniqueProductsPurchased"
    ].lt(0).sum(),
}, name="Errors").to_frame()
product_breadth_checks

### 4.7 Product breadth per historical invoice

`ProductsPerInvoice` divides cumulative distinct products by historical purchase frequency. It measures product breadth after directly normalizing for the number of invoices. A higher value means that the customer has explored more different products per observed invoice.

In [ ]:
snapshot_features["ProductsPerInvoice"] = (
    snapshot_features["UniqueProductsPurchased"]
    / snapshot_features["PurchaseFrequency"]
)

display(snapshot_features["ProductsPerInvoice"].describe())
print("Missing values:", snapshot_features["ProductsPerInvoice"].isna().sum())

### 4.8 United Kingdom indicator

`IsUK` indicates whether the customer's latest valid historical invoice at the reference date is recorded in the United Kingdom. The latest known country is used rather than a country observed in the future. This binary representation follows the earlier EDA finding that individual non-UK countries have limited sample sizes.

In [ ]:
geographic_features = (
    snapshot_purchase_history
    .sort_values(["Customer ID", "ReferenceDate", "PurchaseDate", "Invoice"])
    .drop_duplicates(["Customer ID", "ReferenceDate"], keep="last")
    [["Customer ID", "ReferenceDate", "Country"]]
)
geographic_features["IsUK"] = (
    geographic_features["Country"].eq("United Kingdom").astype("int8")
)
snapshot_features = snapshot_features.merge(
    geographic_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)

display(snapshot_features["IsUK"].value_counts())
print("Missing values:", snapshot_features["IsUK"].isna().sum())

### 4.9 Previous cancellation indicator

`HasCancellation` equals 1 when at least one cancellation invoice is known strictly before the snapshot reference date. It includes cancellation activity regardless of whether the cancellation fully reversed a matched purchase. Future cancellations are excluded.

In [ ]:
snapshot_cancellation_history = (
    model_snapshots[["Customer ID", "ReferenceDate"]]
    .merge(cancellation_events, on="Customer ID")
)
snapshot_cancellation_history = snapshot_cancellation_history.loc[
    snapshot_cancellation_history["CancellationDate"].lt(
        snapshot_cancellation_history["ReferenceDate"]
    )
]
cancellation_features = (
    snapshot_cancellation_history
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(CancellationCount=("Invoice", "nunique"))
)
cancellation_features["HasCancellation"] = 1
snapshot_features = snapshot_features.merge(
    cancellation_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)
snapshot_features[["CancellationCount", "HasCancellation"]] = (
    snapshot_features[["CancellationCount", "HasCancellation"]]
    .fillna(0)
    .astype("int64")
)

display(snapshot_features["HasCancellation"].value_counts())
print("Missing values:", snapshot_features["HasCancellation"].isna().sum())

### 4.10 Purchases during the previous 30 days

`PurchasesLast30Days` counts distinct valid invoices in the half-open historical window `[ReferenceDate - 30 days, ReferenceDate)`. Unlike cumulative purchase frequency, it measures current purchasing momentum. Fully reversed invoices already known at the reference date remain excluded through the cancellation-aware history table.

In [ ]:
recent_purchase_history = snapshot_purchase_history.loc[
    snapshot_purchase_history["PurchaseDate"].ge(
        snapshot_purchase_history["ReferenceDate"]
        - pd.Timedelta(days=30)
    )
]
recent_purchase_features = (
    recent_purchase_history
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(PurchasesLast30Days=("Invoice", "nunique"))
)
snapshot_features = snapshot_features.merge(
    recent_purchase_features,
    on=["Customer ID", "ReferenceDate"],
    how="left",
    validate="one_to_one",
)
snapshot_features["PurchasesLast30Days"] = (
    snapshot_features["PurchasesLast30Days"].fillna(0).astype("int64")
)

display(snapshot_features["PurchasesLast30Days"].describe())
print("Missing values:", snapshot_features["PurchasesLast30Days"].isna().sum())

### 4.11 Explicit churn-risk window

`IsInChurnRiskWindow` equals 1 when current recency is at least 30 days. With a 60-day churn threshold and a 30-day prediction horizon, this is the point at which an active customer can reach the inactivity deadline during the upcoming outcome window. The feature is an explicit, interpretable non-linear transformation of recency.

In [ ]:
snapshot_features["IsInChurnRiskWindow"] = (
    snapshot_features["RecencyDays"].ge(30).astype("int8")
)
risk_window_checks = pd.Series({
    "Missing indicators": snapshot_features[
        "IsInChurnRiskWindow"
    ].isna().sum(),
    "Mismatches with no purchase in the previous 30 days": (
        snapshot_features["IsInChurnRiskWindow"]
        != snapshot_features["PurchasesLast30Days"].eq(0)
    ).sum(),
}, name="Errors").to_frame()
risk_window_checks

## 5. RFM relationship with the target

Only active development snapshots are used here. These are descriptive univariate relationships, not final estimates of model performance. Repeated snapshots from the same customer are correlated, and all final comparisons must use the purged temporal folds.

In [ ]:
rfm_columns = [
    "RecencyDays", "PurchaseFrequency",
    "MonetaryValue", "AverageInvoiceValue",
]
target_column = "WillChurnNext30Days"
churn_development = snapshot_features.loc[
    snapshot_features["FinalSplit"].eq("development")
].copy()

print("Development modeling rows:", len(churn_development))
print("Target rate:", round(churn_development[target_column].mean() * 100, 2), "%")
display(
    churn_development.groupby(target_column)[rfm_columns].median().T
)

### 5.1 Recency and the business boundary

The x-axis uses fixed day intervals, not learned quantiles. With a 60-day churn rule and a 30-day forecast horizon, a customer whose current recency is below 30 days normally cannot reach the time-based boundary during the horizon. A future full reversal can create a small number of exceptions, which are reported separately. `RecencyDays >= 30` is therefore an essential deterministic baseline for the later model notebook.

In [ ]:
churn_development["RecencyBand"] = pd.cut(
    churn_development["RecencyDays"],
    bins=[0, 15, 30, 45, 60.000001],
    labels=["0-14 days", "15-29 days", "30-44 days", "45-59 days"],
    right=False,
)
recency_target = (
    churn_development.groupby("RecencyBand", observed=True)[target_column]
    .agg(Snapshots="size", ChurnsNext30Days="sum", ChurnRate="mean")
)
recency_target["ChurnRate"] *= 100
display(recency_target.round(2))

plt.figure(figsize=(8, 4))
bars = plt.bar(recency_target.index.astype(str), recency_target["ChurnRate"])
plt.title("Churn during the next 30 days by current recency")
plt.xlabel("Recency at reference date")
plt.ylabel("Churn rate (%)")
plt.bar_label(bars, fmt="%.1f%%")
plt.tight_layout()
plt.show()

at_risk_from_recency = churn_development["RecencyDays"].ge(30)
print("Target positives below 30 days of recency:", (
    ~at_risk_from_recency & churn_development[target_column]
).sum())
recency_baseline_table = pd.crosstab(
    at_risk_from_recency.rename("RecencyAtLeast30Days"),
    churn_development[target_column],
    margins=True,
)
true_positives = (at_risk_from_recency & churn_development[target_column]).sum()
print("Baseline precision:", round(true_positives / at_risk_from_recency.sum(), 3))
print("Baseline recall:", round(true_positives / churn_development[target_column].sum(), 3))
print("Baseline accuracy:", round((at_risk_from_recency == churn_development[target_column]).mean(), 3))
recency_baseline_table

### 5.2 Monotonic association

Spearman correlation measures the direction of a monotonic association. It does not prove causality or independent predictive value. Frequency and monetary value may describe overlapping aspects of the same purchase history.

In [ ]:
rfm_spearman = (
    churn_development[rfm_columns + [target_column]]
    .corr(method="spearman")[target_column]
    .drop(target_column)
    .sort_values()
)
plt.figure(figsize=(8, 4))
bars = plt.barh(rfm_spearman.index, rfm_spearman.values)
plt.axvline(0, color="black", linewidth=1)
plt.title("Spearman association with churn in the next 30 days")
plt.xlabel("Spearman correlation")
plt.bar_label(bars, fmt="%.2f")
plt.tight_layout()
plt.show()
rfm_spearman

### 5.3 Frequency and value groups

For each non-recency feature, the development observations are divided into up to five groups containing similar numbers of snapshots. `Q1` contains the lowest observed values. The accompanying table reports the actual minimum, median, and maximum behind each x-axis group so the chart remains interpretable.

In [ ]:
quantile_tables = {}
for feature in ["PurchaseFrequency", "MonetaryValue", "AverageInvoiceValue"]:
    groups = pd.qcut(churn_development[feature], q=5, duplicates="drop")
    table = (
        churn_development.assign(FeatureGroup=groups)
        .groupby("FeatureGroup", observed=True)
        .agg(
            Snapshots=(feature, "size"),
            Minimum=(feature, "min"),
            Median=(feature, "median"),
            Maximum=(feature, "max"),
            ChurnRate=(target_column, "mean"),
        )
        .reset_index(drop=True)
    )
    table.index = [f"Q{i}" for i in range(1, len(table) + 1)]
    table["ChurnRate"] *= 100
    quantile_tables[feature] = table
    display(feature, table.round(2))

    plt.figure(figsize=(7, 4))
    bars = plt.bar(table.index, table["ChurnRate"])
    plt.title(f"Churn in the next 30 days by {feature}")
    plt.xlabel(f"{feature} group, low to high")
    plt.ylabel("Churn rate (%)")
    plt.bar_label(bars, fmt="%.1f%%")
    plt.tight_layout()
    plt.show()

### 5.4 Preliminary findings

Recency dominates the first feature set because it is directly connected to the operational deadline. Only 8 of 8,738 development churn events occur below 30 days of current recency, all through the future full-cancellation mechanism. The transparent rule `RecencyDays >= 30` therefore reaches 99.9% recall, 82.9% precision, and 93.4% accuracy. A machine-learning model is useful only if it improves on this baseline, especially by identifying which customers above 30 days of recency will still purchase before their deadline.

Frequency and monetary value retain moderate negative associations with future churn, while average invoice value is much weaker. These variables remain candidates, but their value must be measured incrementally on the temporal folds. The next justified feature is recency relative to the customer's typical purchase interval.

## 6. Export the first feature table

The processed table contains active customer-reference observations, audit columns, the target, and the first RFM features. Later modeling code must select predictive columns explicitly and exclude identifiers, dates, current-status fields, outcome dates, and the target.

In [ ]:
feature_columns = [
    "RecencyDays",
    "ObservedTenureDays",
    "PurchaseFrequency",
    "MonetaryValue",
    "AverageInvoiceValue",
    "RecencyToMedianGapRatio",
    "ChristmasInvoiceShare",
    "UniqueProductsPurchased",
    "ProductsPerInvoice",
    "IsUK",
    "HasCancellation",
    "PurchasesLast30Days",
    "IsInChurnRiskWindow",
]
forbidden_feature_columns = [
    "Customer ID", "ReferenceDate", "LastPurchaseDate", "Country",
    "ChurnDeadline", "ChurnEventDate", "OutcomeEndDate",
    "LabelEndDate", "IsChurnedAtReference",
    "WillChurnNext30Days", "FinalSplit",
]

export_checks = pd.Series({
    "Missing base feature values": snapshot_features[[
        "RecencyDays", "ObservedTenureDays", "PurchaseFrequency",
        "MonetaryValue", "AverageInvoiceValue",
    ]].isna().any(axis=1).sum(),
    "Missing rhythm among eligible histories": snapshot_features.loc[
        snapshot_features["ObservedPurchaseDays"].ge(2),
        "RecencyToMedianGapRatio",
    ].isna().sum(),
    "Missing Christmas shares": snapshot_features[
        "ChristmasInvoiceShare"
    ].isna().sum(),
    "Missing product counts": snapshot_features[
        "UniqueProductsPurchased"
    ].isna().sum(),
    "Missing product breadth ratios": snapshot_features[
        "ProductsPerInvoice"
    ].isna().sum(),
    "Missing UK indicators": snapshot_features[
        "IsUK"
    ].isna().sum(),
    "Missing cancellation indicators": snapshot_features[
        "HasCancellation"
    ].isna().sum(),
    "Missing recent purchase counts": snapshot_features[
        "PurchasesLast30Days"
    ].isna().sum(),
    "Missing risk-window indicators": snapshot_features[
        "IsInChurnRiskWindow"
    ].isna().sum(),
    "Forbidden columns selected as features": len(
        set(feature_columns).intersection(forbidden_feature_columns)
    ),
    "Duplicate modeling rows": snapshot_features.duplicated(
        ["Customer ID", "ReferenceDate"]
    ).sum(),
}, name="Errors").to_frame()
display(export_checks)

output_path = "../data/processed/churn_snapshot_features.parquet"
snapshot_features.to_parquet(output_path, index=False)
print("Saved:", output_path, snapshot_features.shape)

## Next step

The exported table now includes the original RFM features, the purchase-rhythm candidate, and the conservative `ChristmasInvoiceShare`. Notebook 6 evaluates each candidate incrementally on the two temporal folds of approach B. Product categories remain rule-based and auditable; ambiguous seasonal descriptions are not included in the Christmas feature.